# Decision Tree Random Forest #

### Imports ###



In [11]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


### Load Cleaned Dataset ###

In [12]:
script_dir = os.getcwd()
rel_path = "intermediate"

dataset = pd.read_csv(os.path.join(script_dir, rel_path, "CLEAN_data_injection_moulding.csv"))

### Feature Selection ###

Feature selection need not be too agressive. It's important just to check that we're feeding only process/ sensor variables. We will remove features that are almost always constant as well as irrelevant information.

In [13]:
features = [
    'ZSx [s]',        # injection time
    'ACPx [cm³]',     # cushion volume
    'ZDx [s]',        # dosing time
    # 'ZUs [s]',        # cycle time
    'ZEx [s]',        # (keep if it’s meaningful in your process)
    'GEx [kWh]',      # energy consumption
    'H16x [°C]',      # temperature 1
    'H10x [°C]'       # temperature 2
]

X = dataset[features].copy()
y = dataset['ASZ [Sch]'].astype(int).copy()

print("X shape:", X.shape)
print("y positives:", y.sum(), " / ", len(y))


X shape: (4854, 7)
y positives: 163  /  4854


### Splitting Data ###

In [14]:
SEED = 42


def split_data(X, y, test_size=0.2, random_state=SEED):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=SEED)

print("Train size:", X_train.shape, "Test size:", X_test.shape)
print("Train positives:", y_train.sum(), "Test positives:", y_test.sum())

Train size: (3883, 7) Test size: (971, 7)
Train positives: 130 Test positives: 33


### Baseline Model
This essentially does no machine learning and acts as a lower bound for comparison. It always just predicts the most frequent class. 

In [15]:
baseline = DummyClassifier(strategy="most_frequent", random_state=SEED)
baseline.fit(X_train, y_train)

y_pred_base = baseline.predict(X_test)

print("=== BASELINE ===")
print("F1:", f1_score(y_test, y_pred_base))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_base))
print(classification_report(y_test, y_pred_base, digits=3, zero_division=0))

=== BASELINE ===
F1: 0.0
Confusion matrix:
 [[938   0]
 [ 33   0]]
              precision    recall  f1-score   support

           0      0.966     1.000     0.983       938
           1      0.000     0.000     0.000        33

    accuracy                          0.966       971
   macro avg      0.483     0.500     0.491       971
weighted avg      0.933     0.966     0.949       971



### Decision Tree
Our decision trees might struggle with class imbalance because it may prioritize reducing overall error rather than focusing on our minority defect classes. Class weights solve this by assigning higher importance to minority classes during training. That's why we should select our class weight as balanced because defects are rare in our dataset. I also used mild regularization (max depth + min leaf size) to reduce overfitting

In [16]:
dt = DecisionTreeClassifier(
    random_state=SEED,
    class_weight="balanced",
    max_depth=6,
    min_samples_leaf=10
)

dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print("=== DECISION TREE ===")
print("F1:", f1_score(y_test, y_pred_dt))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt, digits=3, zero_division=0))
dt_cr = classification_report(y_test, y_pred_base, digits=3, zero_division=0,output_dict=True)
%store dt_cr

=== DECISION TREE ===
F1: 0.4732824427480916
Confusion matrix:
 [[871  67]
 [  2  31]]
              precision    recall  f1-score   support

           0      0.998     0.929     0.962       938
           1      0.316     0.939     0.473        33

    accuracy                          0.929       971
   macro avg      0.657     0.934     0.718       971
weighted avg      0.975     0.929     0.945       971

Stored 'dt_cr' (dict)


### Decision Tree Feature Importance
This shows which features the tree relied on most. We can check if that makes sense physically (challenge introducted hinted that importance should align with process understanding)

In [17]:
dt_importance = pd.Series(dt.feature_importances_, index=features).sort_values(ascending=False)
dt_importance


H16x [°C]     0.875538
ZSx [s]       0.050554
H10x [°C]     0.029637
ACPx [cm³]    0.025141
ZEx [s]       0.011276
GEx [kWh]     0.007811
ZDx [s]       0.000043
dtype: float64

### Random Forest

As we know, a random forest reduces variance by averaging. We also see the forest’s feature importance ranking and can compare it to the decision tree to see which fits our process understanding.

In [18]:
rf = RandomForestClassifier(
    random_state=SEED,
    n_estimators=500,
    class_weight="balanced_subsample",
    min_samples_leaf=5,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("=== RANDOM FOREST ===")
print("F1:", f1_score(y_test, y_pred_rf))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf, digits=3, zero_division=0))

rf_cr = classification_report(y_test, y_pred_base, digits=3, zero_division=0,output_dict=True)
%store rf_cr

rf_importance = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
rf_importance


=== RANDOM FOREST ===
F1: 0.691358024691358
Confusion matrix:
 [[918  20]
 [  5  28]]
              precision    recall  f1-score   support

           0      0.995     0.979     0.987       938
           1      0.583     0.848     0.691        33

    accuracy                          0.974       971
   macro avg      0.789     0.914     0.839       971
weighted avg      0.981     0.974     0.977       971

Stored 'rf_cr' (dict)


H16x [°C]     0.604882
GEx [kWh]     0.133143
ZSx [s]       0.080299
ZDx [s]       0.053493
H10x [°C]     0.051246
ACPx [cm³]    0.038989
ZEx [s]       0.037947
dtype: float64

In [19]:
seeds = [0, 1, 2, 3, 4, 5, 7, 11, 21, 42]
f1_dt_list, f1_rf_list = [], []

for s in seeds:
    Xtr, Xte, ytr, yte = split_data(X, y, test_size=0.2, random_state=s)

    dt_tmp = DecisionTreeClassifier(
        random_state=s,
        class_weight="balanced",
        max_depth=6,
        min_samples_leaf=10
    )
    dt_tmp.fit(Xtr, ytr)
    f1_dt_list.append(f1_score(yte, dt_tmp.predict(Xte)))

    rf_tmp = RandomForestClassifier(
        random_state=s,
        n_estimators=500,
        class_weight="balanced_subsample",
        min_samples_leaf=5,
        n_jobs=-1
    )
    rf_tmp.fit(Xtr, ytr)
    f1_rf_list.append(f1_score(yte, rf_tmp.predict(Xte)))

print("DT F1 mean/std:", np.mean(f1_dt_list), np.std(f1_dt_list))
print("RF F1 mean/std:", np.mean(f1_rf_list), np.std(f1_rf_list))


DT F1 mean/std: 0.5290368261383062 0.033544738763274025
RF F1 mean/std: 0.7251994818838975 0.042391902279054545


### Compare Models 

Now we compare the models by using the F1 score

In [20]:
results = pd.DataFrame({
    "Model": ["Baseline", "Decision Tree", "Random Forest"],
    "F1": [
        f1_score(y_test, y_pred_base),
        f1_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_rf)
    ]
}).sort_values("F1", ascending=False)

results


,Model,F1
2,Random Forest,0.691358
1,Decision Tree,0.473282
0,Baseline,0.000000
